# Bark (generative TTS)

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes

A refresher on Bark — Suno's open, transformer-based, *fully generative* text-to-audio model.

## 1. What & Why

**Bark** is a text-prompted, transformer-based generative audio model from Suno. Unlike
classic TTS (Tacotron, VITS, Piper) that maps text → mel-spectrogram → waveform through a
phoneme-aware acoustic model, Bark is a **GPT-style next-token predictor over discrete audio
codes**. You give it text; it autoregressively "writes" the audio the way an LLM writes text.

**What you get for that:**
- **Nonverbal & paralinguistic sounds** baked in — laughter `[laughs]`, sighs, hesitations
  (`um`), `[music]`, sound effects — because it learned from raw audio, not phoneme tables.
- **Multilingual** out of the box (~13 languages) with automatic accent/code-switching.
- **Voice presets / "history prompts"** for rough voice cloning and consistency across calls.
- **Singing & music** snippets via `♪ ... ♪`.

**The cost:** it is **not** a low-latency production engine. It's slow (seconds of compute per
sentence, GPU strongly preferred), capped at **~13 seconds per generation**, and **not
deterministic** — the same prompt yields different takes. Reach for Bark when you want
*expressive, characterful, effects-laden* audio and can tolerate offline batch generation.
Reach for VITS/Piper/StyleTTS 2 when you need fast, stable, repeatable speech.

## 2. Mental Model

**Bark is an LLM for audio.** Same recipe as a text LLM, just three stacked transformers and
a neural codec instead of a tokenizer:

```
 text ──► [Semantic model]  ──► semantic tokens   (the "what to say", prosody-aware)
              GPT-style
                  │
                  ▼
          [Coarse acoustic] ──► coarse codec tokens (first EnCodec codebooks ≈ rough audio)
              GPT-style
                  │
                  ▼
          [Fine acoustic]   ──► fine codec tokens  (remaining codebooks ≈ hi-fi detail)
              GPT-style
                  │
                  ▼
          [EnCodec decoder] ──► 24 kHz waveform
```

The key trick (shared with VALL-E, MusicGen): **audio is turned into a sequence of discrete
integers** by a neural codec (Meta's **EnCodec**), so generating audio becomes the same
problem as generating text — predict the next token. The "voice" you hear is just *which
tokens* the model decides to emit, conditioned on the text and an optional voice-preset
history. That's also why it can emit a `[laughs]` mid-sentence: laughter is just another
region of token-space it learned to visit.

## 3. Key Concepts

- **Three-stage cascade.** Semantic → coarse → fine. The semantic model decides *content and
  prosody*; the two acoustic models turn that into *sound*. Errors compound down the stack.
- **EnCodec tokens.** Audio is quantized into a stack of **codebook** indices (residual vector
  quantization). "Coarse" = the first few codebooks (gross structure); "fine" = the rest
  (timbre/detail). Decoding the full stack reconstructs the waveform.
- **History prompt / voice preset.** A saved tuple of `(semantic, coarse, fine)` tokens that
  seeds generation, giving a consistent speaker. Suno ships presets like `v2/en_speaker_6`;
  you can also reuse Bark's own output as a prompt for continuity.
- **Text markups.** Inline cues — `[laughs]`, `[sighs]`, `[music]`, `[clears throat]`,
  `...` (hesitation), CAPS (emphasis), `♪ lyrics ♪` (song). They're *hints*, not guarantees.
- **Temperature.** Separate temps for text (semantic) and waveform (acoustic) stages control
  diversity vs. stability — higher = more varied/risky, lower = flatter but safer.
- **~13 s cap & nondeterminism.** Each call generates a short clip and is stochastic. Long or
  reproducible audio means chunking + seeding + stitching yourself.
- **`small` models & offloading.** `SUNO_USE_SMALL_MODELS=1` and CPU offload let it run on
  modest hardware at a quality/speed cost.

## 4. Setup

Bark itself is a **large download** (several GB of weights) and wants a GPU for usable speed,
so the real-generation cell below is **gated behind an env var** and won't run by default.
The two worked examples that *do* run only need `numpy` (plus `scipy` for a WAV write) and
illustrate the core ideas — discrete audio codes and autoregressive sampling — on the CPU.

```bash
# The real thing (pick one):
pip install git+https://github.com/suno-ai/bark.git   # official repo
# or use the 🤗 Transformers port:
pip install transformers scipy torch

# Run the heavy cell by setting:  RUN_BARK=1  (and ideally a CUDA GPU)
# Run on modest hardware:         SUNO_USE_SMALL_MODELS=1
```

In [1]:
import sys, numpy as np
print("Python", sys.version.split()[0], "| numpy", np.__version__)

# Optional deps for the gated real-Bark cell — we only *report* availability here.
for mod in ("torch", "transformers", "bark", "scipy"):
    try:
        __import__(mod)
        print(f"  {mod:<12} available")
    except ImportError:
        print(f"  {mod:<12} not installed (fine — only needed for real generation)")

Python 3.13.7 | numpy 2.5.0


  torch        available


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  transformers available
  bark         not installed (fine — only needed for real generation)
  scipy        available


## 5. Worked Examples

The first two examples run on CPU with no model download — they make Bark's two core ideas
concrete: **(1)** audio as a stack of discrete codec tokens, and **(2)** autoregressive,
temperature-controlled token sampling (why Bark is expressive *and* nondeterministic). The
third shows the *real* Bark call shape, gated so the notebook still executes top-to-bottom.

### Example 1 — Audio as discrete codec tokens (the EnCodec idea)

Bark never predicts raw samples; it predicts **integers** that a codec decodes back to sound.
Here's the whole idea in miniature: take a waveform, **quantize** it against a tiny learned
"codebook" (residual VQ with two codebooks ≈ Bark's coarse/fine split), and **decode** it
back. The reconstruction is lossy but recognizable — exactly the trade Bark's GPTs exploit by
generating those small integers instead of 24 000 floats per second.

In [2]:
import numpy as np
rng = np.random.default_rng(0)

# A toy 1-D "audio frame" signal.
t = np.linspace(0, 1, 256)
signal = np.sin(2 * np.pi * 3 * t) + 0.3 * np.sin(2 * np.pi * 11 * t)

# --- A 2-level residual vector quantizer (RVQ): two codebooks of 8 entries each. ---
def make_codebook(values, k=8):
    # "Learn" centroids by quantiles — stand-in for a trained EnCodec codebook.
    return np.quantile(values, np.linspace(0, 1, k))

def quantize(x, codebook):
    idx = np.abs(x[:, None] - codebook[None, :]).argmin(axis=1)  # nearest entry
    return idx, codebook[idx]

cb_coarse = make_codebook(signal)                 # codebook 1: gross structure
res = signal - quantize(signal, cb_coarse)[1]      # residual after coarse pass
cb_fine = make_codebook(res)                       # codebook 2: the leftover detail

# Encode: signal -> two streams of integer tokens (this is what Bark generates).
coarse_idx, coarse_val = quantize(signal, cb_coarse)
fine_idx,   fine_val   = quantize(res,    cb_fine)

# Decode: tokens -> waveform (this is EnCodec's decoder, in miniature).
recon = coarse_val + fine_val

print("First 12 coarse tokens:", coarse_idx[:12])
print("First 12 fine tokens:  ", fine_idx[:12])
rmse_coarse = np.sqrt(np.mean((signal - coarse_val) ** 2))
rmse_full   = np.sqrt(np.mean((signal - recon) ** 2))
print(f"\nReconstruction RMSE  coarse-only: {rmse_coarse:.3f}   coarse+fine: {rmse_full:.3f}")
print(f"Compression: 256 floats -> 256 token-pairs drawn from a {len(cb_coarse)}-entry codebook")

First 12 coarse tokens: [4 4 4 4 5 5 5 6 6 6 6 6]
First 12 fine tokens:   [0 1 5 6 1 4 5 2 3 3 3 2]

Reconstruction RMSE  coarse-only: 0.107   coarse+fine: 0.027
Compression: 256 floats -> 256 token-pairs drawn from a 8-entry codebook


Two codebooks already cut the error noticeably (coarse gets the shape; fine cleans it up) —
the same coarse→fine division of labor Bark's two acoustic transformers learn. Real EnCodec
uses 8 codebooks of 1024 entries at 24 kHz, but the principle is identical: **predicting a few
small integers per frame is tractable for a GPT; predicting raw samples is not.**

### Example 2 — Autoregressive sampling with temperature (why Bark varies)

Each Bark stage is a GPT that **samples** the next token from a softmax distribution. That's
why output is expressive but never identical run-to-run, and why temperature is the main knob.
Below is a minimal next-token sampler over a toy 6-token "vocabulary" (imagine tokens like
`word`, `[laughs]`, `<pause>`). Watch how temperature reshapes the same logits from
near-deterministic to wild.

In [3]:
import numpy as np

vocab = ["the", "cat", "[laughs]", "<pause>", "um", "♪"]
logits = np.array([3.0, 2.2, 0.8, 0.5, 0.3, -0.5])  # model's raw preferences

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def sample(logits, temperature, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    p = softmax(logits / temperature)
    draws = rng.choice(len(logits), size=n, p=p)
    counts = np.bincount(draws, minlength=len(logits)) / n
    return p, counts

for temp in (0.3, 0.7, 1.5):
    p, freq = sample(logits, temp)
    top = vocab[int(np.argmax(p))]
    spread = -(p * np.log(p)).sum()  # entropy = how "spread out" the choice is
    print(f"temp={temp}:  argmax='{top}'  entropy={spread:.2f}  "
          f"P([laughs])={p[2]:.2f}  P(♪)={p[5]:.3f}")

temp=0.3:  argmax='the'  entropy=0.25  P([laughs])=0.00  P(♪)=0.000
temp=0.7:  argmax='the'  entropy=0.85  P([laughs])=0.03  P(♪)=0.005
temp=1.5:  argmax='the'  entropy=1.48  P([laughs])=0.10  P(♪)=0.043


Low temperature collapses onto the safe, high-probability token (flat, repetitive speech);
high temperature flattens the distribution so rare tokens like `[laughs]` or `♪` actually get
emitted — more characterful, but also where Bark goes off the rails (wrong words, artifacts,
hallucinated sounds). Bark exposes **separate** `text_temp` and `waveform_temp` so you can keep
*content* stable while letting *delivery* breathe — typical defaults ≈ 0.7.

### Example 3 — The real Bark call (gated behind `RUN_BARK`)

This is the actual generation shape using 🤗 Transformers' Bark port. It's skipped unless you
set `RUN_BARK=1`, because it pulls several GB of weights and is slow on CPU. The code is
correct as written — set the env var (ideally on a GPU box) to hear it. Note the inline
`[laughs]` markup and the `voice_preset` for a consistent speaker.

In [4]:
import os

if os.getenv("RUN_BARK"):
    from transformers import AutoProcessor, BarkModel
    import scipy.io.wavfile

    model_id = "suno/bark-small"  # ~2 GB; use "suno/bark" for full quality
    processor = AutoProcessor.from_pretrained(model_id)
    model = BarkModel.from_pretrained(model_id)

    text = "Hello! This is Bark generating speech [laughs] with real emotion."
    inputs = processor(text, voice_preset="v2/en_speaker_6")
    audio = model.generate(**inputs)                       # autoregressive: slow
    wav = audio.cpu().numpy().squeeze()

    sr = model.generation_config.sample_rate              # 24000
    scipy.io.wavfile.write("bark_out.wav", rate=sr, data=wav)
    print(f"Wrote bark_out.wav  ({wav.shape[0] / sr:.1f} s @ {sr} Hz)")
else:
    print("RUN_BARK not set — skipping the multi-GB download.")
    print("To run for real:  RUN_BARK=1 python ...  (and ideally a CUDA GPU)")
    print("Native API alternative:")
    print("    from bark import generate_audio, preload_models, SAMPLE_RATE")
    print("    preload_models()")
    print('    audio = generate_audio("Hello [laughs] world", history_prompt="v2/en_speaker_6")')

RUN_BARK not set — skipping the multi-GB download.
To run for real:  RUN_BARK=1 python ...  (and ideally a CUDA GPU)
Native API alternative:
    from bark import generate_audio, preload_models, SAMPLE_RATE
    preload_models()
    audio = generate_audio("Hello [laughs] world", history_prompt="v2/en_speaker_6")


## 6. Gotchas & Pitfalls

- **Nondeterminism by design.** Same text → different audio every call. For repeatable output,
  set seeds (`set_seed` / torch manual seed) *and* pin the voice preset — and even then expect
  drift. Don't build a test that asserts byte-equality on Bark output.
- **The ~13-second wall.** A single generation is capped. Longer narration = split text into
  sentences, generate each (feeding the previous output as a history prompt for continuity),
  and concatenate with small silences. Bark won't do paragraph-length audio in one shot.
- **Markups are suggestions, not commands.** `[laughs]` often works; `[music]` and SFX are hit
  or miss. The model may ignore a cue, place it oddly, or hallucinate it elsewhere. Generate a
  few takes and pick.
- **Hallucinated / dropped words at high temp.** Push `waveform_temp` too high and you get
  garbled or invented words; too low and it's lifeless. ~0.7 is the usual compromise.
- **Speed & memory.** CPU generation is painfully slow. Use `bark-small` / `SUNO_USE_SMALL_MODELS=1`
  and CPU offloading on small GPUs; budget VRAM for three stacked models plus EnCodec.
- **Not a streaming/low-latency engine.** There's no real-time token streaming to a speaker like
  a production TTS API. It's batch/offline.
- **Licensing & voice cloning.** Suno restricts realistic cloning of specific real people; the
  shipped presets are the sanctioned path. Check the model card before cloning a target voice.
- **Two install paths drift.** The native `suno-ai/bark` API (`generate_audio`, `preload_models`)
  and the 🤗 `BarkModel` API differ — don't mix their docs/snippets.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs. Bark |
|---|---|---|
| **Bark** | Expressive, characterful audio with laughter/SFX/singing; multilingual; quick demos | Slow, ~13 s cap, nondeterministic, GPU-hungry; not production-real-time |
| **VITS / [VITS notebook](vits.ipynb)** | Fast, high-quality, end-to-end neural TTS for a fixed voice | Less spontaneous expressiveness; no built-in nonverbal sounds |
| **StyleTTS 2 / [notebook](styletts2.ipynb)** | SOTA naturalness + style/emotion control, still fast | More setup; not "type `[laughs]`" easy |
| **Piper / [notebook](piper-tts.ipynb)** | On-device, real-time, tiny footprint (Raspberry Pi) | Robotic vs. Bark; no emotion/effects |
| **Coqui XTTS / [notebook](coqui-tts.ipynb)** | Strong zero-shot voice cloning from a short clip | Cloning-focused; less "sound design" range |
| **ElevenLabs / [notebook](elevenlabs.ipynb)** | Best-in-class hosted quality, cloning, low latency | Paid SaaS, closed, data leaves your box |
| **Tacotron 2 / [notebook](tacotron.ipynb)** | Classic mel-spectrogram acoustic model to study the pipeline | Needs a separate vocoder; dated for production |

**Rule of thumb:** want *personality and noises* and can batch offline → **Bark**. Want *fast,
stable, repeatable* speech → VITS/StyleTTS 2/Piper. Want *turn-key hosted quality* → ElevenLabs.
Want to *clone a voice from one clip* → Coqui XTTS / ElevenLabs.

## 8. Resources

- **Bark repo (suno-ai/bark)** — official code, voice-preset library, markup list, README:
  https://github.com/suno-ai/bark
- **🤗 Transformers Bark docs** — the `BarkModel` / `AutoProcessor` API used above, plus
  optimization (half-precision, CPU offload, `bark-small`):
  https://huggingface.co/docs/transformers/main/en/model_doc/bark
- **Model cards** — `suno/bark` and `suno/bark-small` on the Hub (license, languages, limits):
  https://huggingface.co/suno/bark
- **EnCodec paper — "High Fidelity Neural Audio Compression" (Défossez et al., 2022)** — the
  discrete-codec foundation Bark generates into: https://arxiv.org/abs/2210.13438
- **VALL-E paper (Wang et al., 2023)** — the "TTS as language modeling over codec tokens"
  paradigm Bark belongs to: https://arxiv.org/abs/2301.02111
- **Bark voice-preset gallery (community)** — auditioning the shipped speakers:
  https://suno-ai.notion.site/8b8e8749ed514b0cbf3f699013548683